In [ ]:
import sys
from pathlib import Path

# Make the repository folder importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
from pbpmaf import framework, assessment, charts

## 1. The framework

In [ ]:
rows = [
    {"Theme": theme, "Subtheme": f"{tn}.{sn} {sub}", "Characteristics of high maturity": len(chars)}
    for _, tn, theme, sn, sub, chars in framework.iter_subthemes()
]
framework_df = pd.DataFrame(rows)
print(f"{len(framework.THEMES)} themes, {len(framework_df)} subthemes, "
      f"{framework_df['Characteristics of high maturity'].sum()} characteristics of high maturity")
framework_df

In [ ]:
print("Steps to Maturity:", " -> ".join(framework.STEPS))
print("Lower maturity steps:", framework.LOWER_MATURITY_STEPS)

## 2. Try an example assessment

Ratings are stored as Steps to Maturity labels keyed by subtheme (`t1s1` = theme 1,
subtheme 1). There is no scoring, weighting or overall result.

In [ ]:
import random

random.seed(42)
example_ratings = {key: random.choice(framework.STEPS) for key, *_ in framework.iter_subthemes()}
example_details = {
    "organisation": "Example Community Healthcare",
    "region": framework.HEALTH_REGIONS[0],
    "team_function": "PBP steering group",
    "date": pd.Timestamp.today().date(),
    "assessor": "",
}

print("Missing details:", assessment.missing_details(example_details))
print("Missing ratings:", assessment.missing_ratings(example_ratings))

In [ ]:
# What the app shows if the assessment is incomplete
partial = dict(list(example_ratings.items())[:20])
assessment.missing_ratings(partial)

## 3. Areas of lower maturity

In [ ]:
lower = assessment.lower_maturity_areas(example_ratings)
lower[["Theme", "Subtheme", "Steps to Maturity"]]

## 4. Charts by theme

In [ ]:
table = assessment.ratings_table(example_ratings)
first_theme = framework.THEMES[0]["theme"]
charts.theme_chart(table[table["Theme"] == first_theme])

## 5. Priority areas for discussion

Grouped by step (Not at all, then To a Small Extent), in framework order within each group. Nothing is weighted or ranked.

In [ ]:
for step, group in assessment.priority_areas(example_ratings):
    print(f"Rated {step} ({len(group)})")
    display(group[["Theme", "Subtheme"]])

## 6. Heat map

In [ ]:
from IPython.display import HTML

HTML(charts.heatmap_html(table))

## 7. PDF report

Writes `example_report.pdf` next to this notebook so you can open it and check the layout.

In [ ]:
from pbpmaf.report import build_pdf

Path("example_report.pdf").write_bytes(build_pdf(example_details, example_ratings))
print("Saved example_report.pdf")